# Étape 2 — Manipulations : préfixes, littéraux typés et vocabulaires standards

Ce notebook accompagne le `README.md`. On va :
1. charger un fichier Turtle existant,
2. comprendre la différence entre un littéral **texte**, un **entier** et une **date**,
3. utiliser les vocabulaires **FOAF** et **Schema.org** depuis Python.

> Prérequis : `pip install -r requirements.txt`. Exécute les cellules dans l'ordre.

## 1. Charger le fichier Turtle enrichi

Au lieu de construire le graphe à la main (étape 1), on **charge** directement le fichier `personnes_enrichi.ttl` avec `graphe.parse(...)`.

In [ ]:
from rdflib import Graph

graphe = Graph()
graphe.parse("personnes_enrichi.ttl", format="turtle")

print(f"{len(graphe)} triplets charges.")

22 triplets charges.


## 2. Les préfixes côté Python

rdflib fournit des classes prêtes pour les vocabulaires standards : `FOAF` et `XSD`. Pour Schema.org, on déclare notre propre `Namespace`. On crée aussi `EX` pour nos identifiants locaux.

C'est l'équivalent Python des lignes `@prefix` du fichier Turtle.

In [ ]:
from rdflib import Namespace
from rdflib.namespace import FOAF, XSD

EX = Namespace("http://exemple.org/")
SCHEMA = Namespace("https://schema.org/")

print("FOAF.name  ->", FOAF.name)
print("FOAF.age   ->", FOAF.age)
print("SCHEMA.worksFor ->", SCHEMA.worksFor)
print("EX.alice   ->", EX.alice)

FOAF.name  -> http://xmlns.com/foaf/0.1/name
FOAF.age   -> http://xmlns.com/foaf/0.1/age
SCHEMA.worksFor -> https://schema.org/worksFor
EX.alice   -> http://exemple.org/alice


## 3. Lire des propriétés avec `graphe.value(...)`

`graphe.value(sujet, predicat)` renvoie l'objet d'un triplet (pratique quand on sait qu'il n'y en a qu'un). Récupérons le nom et l'âge d'Alice.

In [ ]:
nom_alice = graphe.value(EX.alice, FOAF.name)
age_alice = graphe.value(EX.alice, FOAF.age)

print("Nom :", nom_alice)
print("Age :", age_alice)

Nom : Alice Martin
Age : 32


## 4. Pourquoi le TYPE d'un littéral compte

C'est LE point clé de cette étape. Un littéral a un **type de données** (`datatype`). L'âge a été lu comme un entier `xsd:integer`, donc Python le convertit en `int` natif via `.toPython()`. On peut alors faire des calculs dessus.

À l'inverse, si l'âge avait été stocké comme du texte `"32"`, la comparaison numérique échouerait.

In [ ]:
# datatype : le type RDF du litteral
print("datatype de l'age :", age_alice.datatype)   # -> .../XMLSchema#integer

# toPython() : conversion vers un type Python natif
age_python = age_alice.toPython()
print("Type Python :", type(age_python), "| valeur :", age_python)

# Comme c'est un vrai int, on peut calculer :
print("Dans 5 ans, Alice aura :", age_python + 5, "ans")

datatype de l'age : http://www.w3.org/2001/XMLSchema#integer
Type Python : <class 'int'> | valeur : 32
Dans 5 ans, Alice aura : 37 ans


## 5. Comparaison : un littéral typé vs un littéral texte

On crée deux littéraux pour la même valeur « 30 » : un entier et un texte. On voit qu'ils ne sont **pas** considérés comme égaux, et que seul l'entier permet une comparaison numérique.

In [ ]:
from rdflib import Literal

trente_entier = Literal(30, datatype=XSD.integer)
trente_texte = Literal("30")  # texte par defaut

print("Entier == Texte ?", trente_entier == trente_texte)   # False : types differents
print("L'age d'Alice > 30 ?", age_alice.toPython() > 30)      # comparaison numerique OK

Entier == Texte ? False
L'age d'Alice > 30 ? True


## 6. À toi de jouer : ajouter David

**Exercice :** ajoute David au graphe avec les bons types et vocabulaires :
1. `EX.david` est une `FOAF.Person`.
2. Son nom est « David Leroy » (texte).
3. Son âge est `29` (entier `XSD.integer`).
4. Il travaille chez `EX.globex` (`SCHEMA.worksFor`).

Puis réaffiche le Turtle pour vérifier le rendu.

In [ ]:
from rdflib.namespace import RDF

# TODO : complete les triplets concernant David
# graphe.add((EX.david, RDF.type, FOAF.Person))
# ...


# Affichage du resultat (decommente quand tu as ajoute les triplets)
# print(graphe.serialize(format="turtle"))

<details>
<summary>Afficher la solution</summary>

```python
graphe.add((EX.david, RDF.type, FOAF.Person))
graphe.add((EX.david, FOAF.name, Literal("David Leroy")))
graphe.add((EX.david, FOAF.age, Literal(29, datatype=XSD.integer)))
graphe.add((EX.david, SCHEMA.worksFor, EX.globex))

print(graphe.serialize(format="turtle"))
```

</details>

---

Tu maîtrises les préfixes et les types. Direction l'**étape 3** : interroger ce graphe avec SPARQL.